In [ ]:
from bonafide import AtomBondFeaturizer, LogFileAnalyzer

# Example 1
Featurization of a molecule read in as SMILES string (+ analysis of the BONAFIDE log file).

In [ ]:
f = AtomBondFeaturizer()
f.read_input(input_value="COC([C@@H](N1CCC2=C(C1)C=CS2)C3=CC=CC=C3Cl)=O", namespace="clopidogrel")
f.show_molecule(index_type="atom")

In [ ]:
f.list_atom_features(dimensionality="2D", origin="qmdesc")

In [ ]:
f.featurize_atoms(atom_indices=[4, 12], feature_indices=[485, 488])
f.return_atom_features(atom_indices=[4, 12])

In [ ]:
featurized_mol = f.return_atom_features(atom_indices=[4, 12], output_format="mol_object")[0]
print(featurized_mol)

for atom in featurized_mol.GetAtoms():
    print(f"{atom.GetIdx()}: {atom.GetPropsAsDict()}")

In [ ]:
f.list_bond_features(origin="alfabet")

In [ ]:
f.featurize_bonds(bond_indices="all", feature_indices=[0, 1])

In [ ]:
f.show_molecule(index_type="bond")

In [ ]:
f.return_bond_features(bond_indices="all")

In [ ]:
log_analysis = LogFileAnalyzer("bonafide.log")

In [ ]:
log_analysis.get_level_log_messages(log_level="ERROR")

In [ ]:
print(log_analysis.get_level_log_messages(log_level="WARNING"))

In [ ]:
print(f"Time for atom featurization [s]: {log_analysis.get_total_time_for_atom_featurization()}")
print(f"Time for bond featurization [s]: {log_analysis.get_total_time_for_bond_featurization()}")
print(f"Total runtime of BONAFIDE [s]:   {log_analysis.get_total_runtime()}")

In [ ]:
log_analysis.get_time_for_individual_features()

# Example 2
Featurization of a molecule read in as XYZ file and combination with electronic structure data.

In [ ]:
f = AtomBondFeaturizer()
f.read_input(input_value="example_data/spiro_mol-conf_02.xyz", namespace="clopidogrel", input_format="file", output_directory="example_data_out")

In [ ]:
f.determine_bonds()
f.show_molecule()

In [ ]:
f.list_atom_features(origin="morfeus")

In [ ]:
f.featurize_atoms(atom_indices=[1, 7], feature_indices=[177, 167, 193])

In [ ]:
f.return_atom_features(atom_indices=[1, 7])

In [ ]:
f.print_options(origin="morfeus")

In [ ]:
radii = [2.5, 2.8, 3.1, 3.4, 3.7, 4.0]

for r in radii:
    f.clear_atom_feature_cache()  # Clear feature cache before each iteration to allow the feature to be recalculated, otherwise, it is pulled from the cache
    print(f"Radius: {r}")
    f.set_options(("morfeus.buried_volume.radius", r))
    f.featurize_atoms(atom_indices=7, feature_indices=177)
    print(f.return_atom_features(atom_indices=7, output_format="dict")[7])
    print()

In [ ]:
f.attach_electronic_structure(electronic_structure_data="example_data/spiro_mol-conf_02.fchk")

In [ ]:
f.set_charge(0)
f.set_multiplicity(1)

In [ ]:
f.list_bond_features(dimensionality="3D", origin="multiWFN").head(10)

In [ ]:
f.list_bond_features(origin="rdkit")

In [ ]:
f.featurize_bonds(bond_indices="all", feature_indices=[431, 538])

In [ ]:
f.return_bond_features()

# Example 3
Featurization of a molecule read in as XYZ file and calculation of its electronic structure from scratch.

In [ ]:
f = AtomBondFeaturizer()
f.read_input(input_value="example_data/spiro_mol-conf_02.xyz", namespace="clopidogrel", input_format="file", output_directory="example_data_out")

In [ ]:
f.print_options(origin="psi4")

In [ ]:
f.set_charge(0)
f.set_multiplicity(1)

f.calculate_electronic_structure(engine="psi4", redox="n")

In [ ]:
f.featurize_atoms(atom_indices="all", feature_indices=[370, 264])

In [ ]:
f.determine_bonds()
f.show_molecule()

In [ ]:
f.return_atom_features()

# Example 4
Featurization of an ensemble of conformers read from an SD file (energy data included).

In [ ]:
f = AtomBondFeaturizer()
f.read_input(input_value="example_data/clopidogrel_e.sdf", namespace="clopidogrel", input_format="file", read_energy=True, prune_by_energy=(1, "kj/mol"))

In [ ]:
f.show_molecule(in_3D=True)

In [ ]:
f.set_charge(0)
f.featurize_atoms(atom_indices=[4, 11], feature_indices=[98, 177])

In [ ]:
f.return_atom_features(atom_indices=[4, 11])

In [ ]:
f.return_atom_features(atom_indices=[4, 11], output_format="dict", reduce=True)

# Example 5
Attaching of pre-computed energy data to an conformer ensemble.

In [ ]:
f = AtomBondFeaturizer()
f.read_input(input_value="example_data/clopidogrel_e.sdf", namespace="clopidogrel", input_format="file", read_energy=False)

In [ ]:
f.mol_vault.size  # Number of conformers in the molecule vault

In [ ]:
energy_data = [
    (-763.1596, "eh"),
    (-763.1429, "eh"),  # high-energy conformer
    (-763.1601, "eh"),
    (-763.1552, "eh"),
    (-763.1543, "eh"),
    (-763.1433, "eh"),  # high-energy conformer
    (-763.1590, "eh"),
]  # dummy energy values (not physically meaningful/correct)

f.attach_energy(energy_data=energy_data, state="n", prune_by_energy=(10, "kcal/mol"))

In [ ]:
f.mol_vault.update_boltzmann_weights(temperature=298.15, ignore_invalid=True)  # for demonstration purposes only

In [ ]:
f.show_molecule(in_3D=True)

In [ ]:
f.mol_vault.is_valid

In [ ]:
f.mol_vault.boltzmann_weights

In [ ]:
sum([w for w in f.mol_vault.boltzmann_weights[1] if w])

# Example 6
Featurization of a molecule directly read-in as RDKit molecule object

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole

IPythonConsole.drawOptions.addAtomIndices = True
IPythonConsole.molSize = (400, 400)

In [ ]:
mol = Chem.MolFromSmiles("O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl")
mol = Chem.AddHs(mol)
AllChem.EmbedMultipleConfs(mol, numConfs=3)
mol

In [ ]:
f = AtomBondFeaturizer()
f.read_input(mol, "diclofenac", input_format="mol_object")

In [ ]:
f.show_molecule(index_type="atom")

In [ ]:
f.set_charge(0)
f.featurize_atoms(atom_indices="all", feature_indices=98)

In [ ]:
f.return_atom_features()